# Fake Review Detector: Snowpark model + Ask in English

**Part 1 (cells 1–7):** train a real model with Snowpark that learns the best weights for the 3 AI signals, tested on unseen reviews.

**Part 2 (cells 8–10):** ask questions in plain English with Cortex Analyst (needs file `08_semantic_view.sql` run first).

Run the SQL files 01 → 04 before this notebook.

## 1. Connect to Snowflake

In [ ]:
from snowflake.snowpark.context import get_active_session
from snowflake.snowpark import functions as F

session = get_active_session()
session.use_warehouse('REVIEWS_WH')
session.use_database('REVIEWS_DB')
session.use_schema('PUBLIC')

## 2. Build features with the Snowpark DataFrame API

In [ ]:
# (this runs inside Snowflake as SQL - Python just describes it)
s = session.table('REVIEW_SCORES')

feats = s.select(
    F.col('REVIEW_ID'),
    F.iff(F.col('AI_VERDICT').like('generic%'), 1, 0).alias('AI_FAKE'),
    F.col('MAX_SIMILARITY'),
    F.iff(F.col('SENTIMENT') == 'neutral', 1, 0).alias('IS_NEUTRAL'),
    F.iff(F.col('SENTIMENT') == 'mixed', 1, 0).alias('IS_MIXED'),
    F.iff(((F.col('RATING') >= 4) & (F.col('SENTIMENT') == 'negative')) |
          ((F.col('RATING') <= 2) & (F.col('SENTIMENT') == 'positive')), 1, 0).alias('MISMATCH'),
    F.col('RATING').cast('float').alias('RATING'),
    (F.length(F.col('REVIEW_TEXT')) / 100).alias('LENGTH_100'),
    F.col('SUSPICION_SCORE'),
    F.iff(F.col('LABEL') == 'CG', 1, 0).alias('IS_FAKE'),
)
feats.show()

## 3. Split 80% train / 20% test

In [ ]:
train_sp, test_sp = feats.random_split([0.8, 0.2], seed=42)
train = train_sp.to_pandas()
test = test_sp.to_pandas()
print('train rows:', len(train), '| test rows:', len(test))

## 4. Train the model

In [ ]:
from sklearn.linear_model import LogisticRegression

cols = ['AI_FAKE', 'MAX_SIMILARITY', 'IS_NEUTRAL', 'IS_MIXED', 'MISMATCH', 'RATING', 'LENGTH_100']
model = LogisticRegression(max_iter=1000)
model.fit(train[cols], train['IS_FAKE'])

## 5. Test on unseen reviews: model vs hand-made score

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score

test['PREDICTED'] = model.predict(test[cols])
test['HANDMADE'] = (test['SUSPICION_SCORE'] >= 30).astype(int)

for name in ['PREDICTED', 'HANDMADE']:
    print(f"{name:10} accuracy {accuracy_score(test.IS_FAKE, test[name]):.3f} | "
          f"precision {precision_score(test.IS_FAKE, test[name]):.3f} | "
          f"recall {recall_score(test.IS_FAKE, test[name]):.3f}")

## 6. What did the model learn?

In [ ]:
# positive weight = pushes towards FAKE, negative = towards REAL
for c, w in sorted(zip(cols, model.coef_[0]), key=lambda x: -abs(x[1])):
    print(f"{c:15} {w:+.3f}")

## 7. Save predictions back to Snowflake

In [ ]:
session.write_pandas(
    test[['REVIEW_ID', 'IS_FAKE', 'PREDICTED', 'HANDMADE']],
    'REVIEW_ML_PREDICTIONS',
    auto_create_table=True,
    overwrite=True,
)
session.table('REVIEW_ML_PREDICTIONS').show()

---
# Part 2: Ask in English (Cortex Analyst)

## 8. Imports for the REST call

In [ ]:
import requests

## 9. The ask() function

In [ ]:
def ask(question):
    body = {
        'messages': [{'role': 'user', 'content': [{'type': 'text', 'text': question}]}],
        'semantic_view': 'REVIEWS_DB.PUBLIC.REVIEWS_SEMANTIC',
    }
    # Cortex Analyst is a REST API; the notebook's own login token is used
    r = requests.post(
        f'https://{session.connection.host}/api/v2/cortex/analyst/message',
        headers={'Authorization': f'Snowflake Token="{session.connection.rest.token}"',
                 'Content-Type': 'application/json'},
        json=body, timeout=60,
    )
    r.raise_for_status()
    for part in r.json()['message']['content']:
        if part['type'] == 'text':
            print('💬', part['text'])
        elif part['type'] == 'sql':
            print('\n🧾 SQL it wrote:\n', part['statement'])
            return session.sql(part['statement']).to_pandas()
        elif part['type'] == 'suggestions':
            print('Try asking:', part['suggestions'])

## 10. Ask anything

In [ ]:
ask('Which product category has the most flagged reviews?')

# more to try (one per cell):
# ask('What is the average star rating of flagged vs not flagged reviews?')
# ask('How many fake reviews have positive sentiment?')
# ask('Show the average suspicion score for each category')